# Módulos estándar · asyncio y utilidades

Continuación de *Concurrencia* y *Sincronización de hilos* con `asyncio` a fondo: tareas y `TaskGroup`, límites de tiempo, cancelación con limpieza, colas, semáforos, `gather` con excepciones y `as_completed`. Después, un repaso de módulos pequeños pero muy útiles: `contextlib`, `functools.singledispatch`, `decimal`, `difflib`, `textwrap` y `weakref`. Las esperas se simulan con `asyncio.sleep` de pocos milisegundos y no se imprimen duraciones, que cambian en cada ejecución; los ejercicios comprueban órdenes y resultados. En Jupyter ya hay un bucle de eventos en marcha, así que se usa `await` directamente en las celdas en lugar de `asyncio.run`.

## 1. Tareas y `TaskGroup`

Simular la descarga de los horarios de cinco guaguas con una corrutina `descargar(linea, espera)` que duerme `espera` segundos y devuelve un texto. Lanzarlas como tareas dentro de un `asyncio.TaskGroup` y recoger los resultados al salir del bloque. Comprobar que el tiempo total se parece al de la descarga más lenta y no a la suma, y que si una tarea falla, el grupo cancela las demás y lanza un `ExceptionGroup`.

In [1]:
import asyncio
import time

LINEAS = {"L1": 0.05, "L2": 0.02, "L12": 0.04, "L30": 0.01, "L91": 0.03}


async def descargar(linea, espera):
    await asyncio.sleep(espera)
    if espera < 0:
        raise ValueError(f"espera negativa en {linea}")
    return f"horario {linea}"


inicio = time.perf_counter()
async with asyncio.TaskGroup() as grupo:
    tareas = {linea: grupo.create_task(descargar(linea, t)) for linea, t in LINEAS.items()}
duracion = time.perf_counter() - inicio

resultados = {linea: tarea.result() for linea, tarea in tareas.items()}
print(resultados)
assert max(LINEAS.values()) <= duracion < sum(LINEAS.values())
print("Duró como la más lenta, no como la suma:", True)

{'L1': 'horario L1', 'L2': 'horario L2', 'L12': 'horario L12', 'L30': 'horario L30', 'L91': 'horario L91'}
Duró como la más lenta, no como la suma: True


In [2]:
canceladas = []


async def lenta(linea):
    try:
        await asyncio.sleep(1)
    except asyncio.CancelledError:
        canceladas.append(linea)
        raise


try:
    async with asyncio.TaskGroup() as grupo:
        grupo.create_task(lenta("L1"))
        grupo.create_task(lenta("L2"))
        grupo.create_task(descargar("L99", -0.01))
except* ValueError as eg:
    print("ExceptionGroup con:", [str(e) for e in eg.exceptions])
print("Canceladas por el grupo:", sorted(canceladas))
assert sorted(canceladas) == ["L1", "L2"]

ExceptionGroup con: ['espera negativa en L99']
Canceladas por el grupo: ['L1', 'L2']


## 2. Límite de tiempo con `asyncio.timeout`

Escribir `consultar_con_limite(espera, limite)` que espere una respuesta simulada como mucho `limite` segundos usando `async with asyncio.timeout(...)`, y devuelva `"respuesta"` o `"sin respuesta"` si se agota el tiempo. Probarlo con varias combinaciones y comparar con `asyncio.wait_for`.

In [3]:
async def respuesta(espera):
    await asyncio.sleep(espera)
    return "respuesta"


async def consultar_con_limite(espera, limite):
    try:
        async with asyncio.timeout(limite):
            return await respuesta(espera)
    except TimeoutError:
        return "sin respuesta"


casos = [(0.01, 0.05), (0.08, 0.03), (0.02, 0.1)]
print(f"{'espera':>7} {'límite':>7}  resultado")
for espera, limite in casos:
    print(f"{espera:7.2f} {limite:7.2f}  {await consultar_con_limite(espera, limite)}")

try:
    await asyncio.wait_for(respuesta(0.08), timeout=0.02)
except TimeoutError:
    print("wait_for también lanza TimeoutError")

assert await consultar_con_limite(0.01, 0.05) == "respuesta"
assert await consultar_con_limite(0.08, 0.03) == "sin respuesta"

 espera  límite  resultado
   0.01    0.05  respuesta
   0.08    0.03  sin respuesta
   0.02    0.10  respuesta
wait_for también lanza TimeoutError


## 3. Cancelación y limpieza

Una tarea `vigilar_sensor` lee un sensor en bucle cada pocos milisegundos y guarda las lecturas. Cancelarla desde fuera y asegurarse, con `try`/`finally`, de que cierra la conexión simulada aunque la cancelen. Comprobar que `tarea.cancelled()` es cierto y que la conexión quedó cerrada.

In [4]:
class Conexion:
    def __init__(self):
        self.abierta = True

    def cerrar(self):
        self.abierta = False


async def vigilar_sensor(conexion, lecturas):
    try:
        n = 0
        while True:
            lecturas.append(20 + n % 3)
            n += 1
            await asyncio.sleep(0.005)
    finally:
        conexion.cerrar()


conexion, lecturas = Conexion(), []
tarea = asyncio.create_task(vigilar_sensor(conexion, lecturas))
await asyncio.sleep(0.03)
tarea.cancel()
try:
    await tarea
except asyncio.CancelledError:
    print("La tarea se canceló")

print("Cancelada:", tarea.cancelled(), "| conexión abierta:", conexion.abierta, "| hubo lecturas:", len(lecturas) > 0)
assert tarea.cancelled() and not conexion.abierta and lecturas

La tarea se canceló


Cancelada: True | conexión abierta: False | hubo lecturas: True


## 4. Productor-consumidor con `asyncio.Queue`

Un productor mete en una `asyncio.Queue(maxsize=3)` los pedidos de una pescadería (10 pedidos) y tres consumidores los preparan. Usar `queue.join()` para esperar a que se procesen todos y cancelar después a los consumidores, que esperan indefinidamente. Comprobar que se prepararon todos los pedidos una sola vez.

In [5]:
async def productor(cola, n):
    for i in range(1, n + 1):
        await cola.put(f"pedido-{i:02}")


async def consumidor(nombre, cola, hechos):
    while True:
        pedido = await cola.get()
        await asyncio.sleep(0.005)
        hechos.append((nombre, pedido))
        cola.task_done()


cola, hechos = asyncio.Queue(maxsize=3), []
consumidores = [asyncio.create_task(consumidor(f"dependiente{i}", cola, hechos)) for i in range(1, 4)]
await productor(cola, 10)
await cola.join()
for c in consumidores:
    c.cancel()
await asyncio.gather(*consumidores, return_exceptions=True)

pedidos = sorted(p for _, p in hechos)
print("Preparados:", len(pedidos), "| distintos:", len(set(pedidos)))
print("Dependientes que trabajaron:", len({n for n, _ in hechos}))
assert pedidos == [f"pedido-{i:02}" for i in range(1, 11)]

Preparados: 10 | distintos: 10
Dependientes que trabajaron: 3


## 5. Limitar la concurrencia con `asyncio.Semaphore`

Simular 20 consultas a un servidor de mareas que solo admite 4 a la vez. Envolver cada consulta en un `async with semaforo` y medir cuántas hay activas como máximo. Comprobar que nunca pasan de 4 y que sin semáforo llegarían a 20.

In [6]:
async def consultas(n, semaforo=None):
    activas = maximo = 0

    async def una(i):
        nonlocal activas, maximo
        activas += 1
        maximo = max(maximo, activas)
        await asyncio.sleep(0.01)
        activas -= 1
        return i * i

    async def con_limite(i):
        if semaforo is None:
            return await una(i)
        async with semaforo:
            return await una(i)

    resultados = await asyncio.gather(*(con_limite(i) for i in range(n)))
    return resultados, maximo


res, maximo = await consultas(20, asyncio.Semaphore(4))
_, maximo_libre = await consultas(20)
print("Máximo con semáforo:", maximo, "| sin semáforo:", maximo_libre)
assert maximo == 4 and maximo_libre == 20 and res == [i * i for i in range(20)]

Máximo con semáforo: 4 | sin semáforo: 20


## 6. `gather` con `return_exceptions`

Comprobar el precio del billete de barco a cinco islas; dos de las consultas fallan. Con `gather` normal, la primera excepción se propaga; con `return_exceptions=True` se obtienen todos los resultados mezclados con las excepciones, en el mismo orden que las corrutinas. Separar los precios correctos de los errores.

In [7]:
async def precio_barco(isla):
    await asyncio.sleep(0.005)
    precios = {"La Gomera": 41.5, "La Palma": 58.0, "Lanzarote": 49.9}
    if isla not in precios:
        raise LookupError(f"sin ruta a {isla}")
    return precios[isla]


islas = ["La Gomera", "El Hierro", "La Palma", "La Graciosa", "Lanzarote"]
try:
    await asyncio.gather(*(precio_barco(i) for i in islas))
except LookupError as e:
    print("gather normal:", e)

resultados = await asyncio.gather(*(precio_barco(i) for i in islas), return_exceptions=True)
correctos = {i: r for i, r in zip(islas, resultados) if not isinstance(r, Exception)}
errores = {i: str(r) for i, r in zip(islas, resultados) if isinstance(r, Exception)}
print("Precios:", correctos)
print("Errores:", errores)
assert list(correctos) == ["La Gomera", "La Palma", "Lanzarote"] and len(errores) == 2

gather normal: sin ruta a El Hierro
Precios: {'La Gomera': 41.5, 'La Palma': 58.0, 'Lanzarote': 49.9}
Errores: {'El Hierro': 'sin ruta a El Hierro', 'La Graciosa': 'sin ruta a La Graciosa'}


## 7. Resultados según llegan con `as_completed`

Lanzar cuatro búsquedas de vuelos con esperas distintas y procesar cada resultado en cuanto termine con `asyncio.as_completed`, en lugar de esperar a todos. Comprobar que el orden de llegada es el de las esperas y que `gather` devuelve, en cambio, el orden de lanzamiento.

In [8]:
async def buscar_vuelo(aerolinea, espera):
    await asyncio.sleep(espera)
    return aerolinea


esperas = {"Binter": 0.04, "Iberia": 0.01, "Vueling": 0.06, "Ryanair": 0.025}
llegada = []
for futuro in asyncio.as_completed([buscar_vuelo(a, t) for a, t in esperas.items()]):
    aerolinea = await futuro
    llegada.append(aerolinea)
    print("Llega:", aerolinea)

lanzamiento = await asyncio.gather(*(buscar_vuelo(a, t) for a, t in esperas.items()))
print("gather:", lanzamiento)
assert llegada == sorted(esperas, key=esperas.get)
assert lanzamiento == list(esperas)

Llega: Iberia
Llega: Ryanair
Llega: Binter


Llega: Vueling
gather: ['Binter', 'Iberia', 'Vueling', 'Ryanair']


## 8. `contextlib`: limpieza con `contextmanager` y `ExitStack`

Escribir con `@contextmanager` un gestor `abrir_registro(nombre, diario)` que anote en `diario` cuándo se abre y se cierra un recurso, también si el bloque lanza una excepción. Después, abrir un número variable de recursos con `ExitStack`, que los cierra en orden inverso, y comprobar que si falla la apertura del tercero, los dos primeros se cierran igualmente.

In [9]:
from contextlib import ExitStack, contextmanager


@contextmanager
def abrir_registro(nombre, diario):
    if nombre.startswith("roto"):
        raise OSError(f"no se puede abrir {nombre}")
    diario.append(f"abre {nombre}")
    try:
        yield nombre.upper()
    finally:
        diario.append(f"cierra {nombre}")


diario = []
try:
    with abrir_registro("caja", diario):
        raise RuntimeError("fallo dentro del bloque")
except RuntimeError as e:
    print("Excepción:", e)
print(diario)
assert diario == ["abre caja", "cierra caja"]

Excepción: fallo dentro del bloque
['abre caja', 'cierra caja']


In [10]:
diario = []
with ExitStack() as pila:
    recursos = [pila.enter_context(abrir_registro(n, diario)) for n in ["a", "b", "c"]]
    print("Dentro:", recursos)
print(diario)
assert diario == ["abre a", "abre b", "abre c", "cierra c", "cierra b", "cierra a"]

diario = []
try:
    with ExitStack() as pila:
        for n in ["a", "b", "roto", "d"]:
            pila.enter_context(abrir_registro(n, diario))
except OSError as e:
    print("Error:", e)
print(diario)
assert diario == ["abre a", "abre b", "cierra b", "cierra a"]

Dentro: ['A', 'B', 'C']
['abre a', 'abre b', 'abre c', 'cierra c', 'cierra b', 'cierra a']
Error: no se puede abrir roto
['abre a', 'abre b', 'cierra b', 'cierra a']


## 9. `suppress` y `redirect_stdout`

Una función antigua `informe_ventas()` escribe con `print` en lugar de devolver el texto. Capturar su salida en una cadena con `redirect_stdout` y un `io.StringIO` para analizarla. Usar `suppress` para borrar un fichero que quizá no exista sin un `try`/`except` vacío.

In [11]:
import io
from contextlib import redirect_stdout, suppress
from pathlib import Path


def informe_ventas():
    for producto, kilos in [("plátanos", 120), ("papas", 85), ("tomates", 64)]:
        print(f"{producto:<10}{kilos:>5} kg")


buffer = io.StringIO()
with redirect_stdout(buffer):
    informe_ventas()
texto = buffer.getvalue()
total = sum(int(linea.split()[1]) for linea in texto.splitlines())
print("Líneas capturadas:", len(texto.splitlines()), "| total:", total, "kg")
assert total == 269

with suppress(FileNotFoundError):
    Path("no_existe_asyncio.tmp").unlink()
print("Borrado sin error aunque el fichero no existía")

Líneas capturadas: 3 | total: 269 kg
Borrado sin error aunque el fichero no existía


## 10. Funciones genéricas con `functools.singledispatch`

Escribir `describir(valor)` que se comporte distinto según el tipo del argumento: enteros, decimales, cadenas, listas y fechas, registrando cada versión con `@describir.register` y anotaciones de tipo. Los tipos no registrados usan la versión base. Comprobar que `bool`, al ser subclase de `int`, usa la versión de `int` salvo que se registre la suya.

In [12]:
from datetime import date
from functools import singledispatch


@singledispatch
def describir(valor):
    return f"objeto {type(valor).__name__}"


@describir.register
def _(valor: int):
    return f"entero {'par' if valor % 2 == 0 else 'impar'}"


@describir.register
def _(valor: float):
    return f"decimal con {valor:.2f}"


@describir.register
def _(valor: str):
    return f"texto de {len(valor)} caracteres"


@describir.register
def _(valor: list):
    return "lista de [" + ", ".join(describir(v) for v in valor) + "]"


@describir.register
def _(valor: date):
    return f"fecha {valor:%d/%m/%Y}"


for v in [7, 3.14159, "Teide", [2, "a"], date(2024, 5, 30), True, None]:
    print(f"{v!r:>25} -> {describir(v)}")

assert describir(True) == "entero impar"
describir.register(bool, lambda v: "booleano")
assert describir(True) == "booleano" and describir(None) == "objeto NoneType"

                        7 -> entero impar
                  3.14159 -> decimal con 3.14
                  'Teide' -> texto de 5 caracteres
                 [2, 'a'] -> lista de [entero par, texto de 1 caracteres]
datetime.date(2024, 5, 30) -> fecha 30/05/2024
                     True -> entero impar
                     None -> objeto NoneType


## 11. `decimal`: dinero, precisión y redondeo

Repartir 100 € entre tres socios: con `float` la suma de las partes redondeadas no siempre cuadra. Con `Decimal`, calcular las partes a céntimos y asignar el céntimo sobrante. Mostrar la diferencia entre `0.1 + 0.2` en `float` y en `Decimal`, cambiar la precisión con `localcontext` y comparar los modos de redondeo `ROUND_HALF_UP`, `ROUND_HALF_EVEN`, `ROUND_DOWN` y `ROUND_CEILING` sobre varios importes.

In [13]:
from decimal import ROUND_CEILING, ROUND_DOWN, ROUND_HALF_EVEN, ROUND_HALF_UP, Decimal, localcontext

print("float:  ", 0.1 + 0.2, "==", 0.3, "->", 0.1 + 0.2 == 0.3)
print("Decimal:", Decimal("0.1") + Decimal("0.2"), "->", Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))
print("Decimal(0.1) desde float:", Decimal(0.1))

CENTIMO = Decimal("0.01")


def repartir(total, partes):
    base = (total / partes).quantize(CENTIMO, rounding=ROUND_DOWN)
    sobrante = int((total - base * partes) / CENTIMO)
    return [base + CENTIMO if i < sobrante else base for i in range(partes)]


reparto = repartir(Decimal("100"), 3)
print("Reparto:", reparto, "suma", sum(reparto))
assert sum(reparto) == Decimal("100")
print("Con float:", [round(100 / 3, 2)] * 3, "suma", sum([round(100 / 3, 2)] * 3))

float:   0.30000000000000004 == 0.3 -> False
Decimal: 0.3 -> True
Decimal(0.1) desde float: 0.1000000000000000055511151231257827021181583404541015625
Reparto: [Decimal('33.34'), Decimal('33.33'), Decimal('33.33')] suma 100.00
Con float: [33.33, 33.33, 33.33] suma 99.99


In [14]:
with localcontext() as ctx:
    ctx.prec = 40
    print("1/7 con 40 cifras:", Decimal(1) / Decimal(7))
print("1/7 con 28 (por defecto):", Decimal(1) / Decimal(7))

modos = [ROUND_HALF_UP, ROUND_HALF_EVEN, ROUND_DOWN, ROUND_CEILING]
importes = ["2.345", "2.355", "-2.345", "2.341"]
print(f"{'importe':>8}" + "".join(f"{m:>17}" for m in modos))
for imp in importes:
    print(f"{imp:>8}" + "".join(f"{str(Decimal(imp).quantize(CENTIMO, rounding=m)):>17}" for m in modos))

assert Decimal("2.345").quantize(CENTIMO, ROUND_HALF_EVEN) == Decimal("2.34")
assert Decimal("2.355").quantize(CENTIMO, ROUND_HALF_EVEN) == Decimal("2.36")
assert Decimal("-2.345").quantize(CENTIMO, ROUND_CEILING) == Decimal("-2.34")

1/7 con 40 cifras: 0.1428571428571428571428571428571428571429
1/7 con 28 (por defecto): 0.1428571428571428571428571429
 importe    ROUND_HALF_UP  ROUND_HALF_EVEN       ROUND_DOWN    ROUND_CEILING
   2.345             2.35             2.34             2.34             2.35
   2.355             2.36             2.36             2.35             2.36
  -2.345            -2.35            -2.34            -2.34            -2.34
   2.341             2.34             2.34             2.34             2.35


## 12. `difflib`: diferencias y nombres parecidos

Comparar dos versiones de una receta de mojo picón con `difflib.unified_diff` y mostrar el parche. Calcular la similitud de las dos versiones con `SequenceMatcher.ratio()`. Después, sugerir el municipio correcto cuando alguien lo escribe mal con `get_close_matches`.

In [15]:
import difflib

v1 = ["4 dientes de ajo", "1 pimienta palmera", "1 cucharadita de comino", "sal gruesa", "100 ml de aceite", "2 cucharadas de vinagre"]
v2 = ["5 dientes de ajo", "1 pimienta palmera", "1 cucharadita de comino", "1 cucharadita de pimentón", "sal gruesa", "100 ml de aceite", "2 cucharadas de vinagre"]

for linea in difflib.unified_diff(v1, v2, "mojo_v1", "mojo_v2", lineterm="", n=1):
    print(linea)

similitud = difflib.SequenceMatcher(None, v1, v2).ratio()
print(f"Similitud por líneas: {similitud:.3f}")
assert similitud == 2 * 5 / (len(v1) + len(v2))

--- mojo_v1
+++ mojo_v2
@@ -1,4 +1,5 @@
-4 dientes de ajo
+5 dientes de ajo
 1 pimienta palmera
 1 cucharadita de comino
+1 cucharadita de pimentón
 sal gruesa
Similitud por líneas: 0.769


In [16]:
municipios = ["Arucas", "Agaete", "Gáldar", "Telde", "Teror", "Tejeda", "Mogán", "Ingenio", "Agüimes", "Firgas", "Moya"]
for escrito in ["Arukas", "Teld", "Galdar", "Tejera", "Madrid"]:
    print(f"{escrito:<8} -> {difflib.get_close_matches(escrito, municipios, n=2, cutoff=0.6)}")

assert difflib.get_close_matches("Arukas", municipios, n=1) == ["Arucas"]
assert difflib.get_close_matches("Madrid", municipios) == []

Arukas   -> ['Arucas']
Teld     -> ['Telde', 'Tejeda']
Galdar   -> ['Gáldar']
Tejera   -> ['Tejeda']
Madrid   -> []


## 13. `textwrap`: una ficha en columnas

Componer la ficha de un sendero en dos columnas de 28 caracteres: a la izquierda los datos y a la derecha la descripción, ajustada con `textwrap.wrap`. Rellenar la columna más corta para que las filas encajen y sangrar la nota final con `indent` solo en las líneas no vacías (con su parámetro `predicate`). Comprobar que ninguna línea supera el ancho.

In [17]:
import textwrap
from itertools import zip_longest

ANCHO = 28
datos = ["Sendero: Roque Nublo", "Isla: Gran Canaria", "Distancia: 5,1 km", "Desnivel: 180 m", "Dificultad: baja"]
descripcion = ("Desde La Goleta, una pista empedrada sube entre pinos hasta la base del roque, uno de los "
               "símbolos de la isla, con vistas al Teide en días despejados.")
derecha = textwrap.wrap(descripcion, ANCHO)
filas = [f"{izq:<{ANCHO}} | {der}" for izq, der in zip_longest(datos, derecha, fillvalue="")]
print("\n".join(filas))

nota = "Llevar agua.\n\nEvitar los días de viento fuerte."
print(textwrap.indent(nota, "  > ", predicate=lambda linea: linea.strip()))
assert all(len(linea) <= ANCHO for linea in derecha)
assert " ".join(derecha) == descripcion

Sendero: Roque Nublo         | Desde La Goleta, una pista
Isla: Gran Canaria           | empedrada sube entre pinos
Distancia: 5,1 km            | hasta la base del roque, uno
Desnivel: 180 m              | de los símbolos de la isla,
Dificultad: baja             | con vistas al Teide en días
                             | despejados.
  > Llevar agua.

  > Evitar los días de viento fuerte.


## 14. `weakref`: caché con `WeakValueDictionary` y `finalize`

Crear una caché de imágenes con `weakref.WeakValueDictionary`: mientras alguien use una imagen, pedirla de nuevo devuelve el mismo objeto sin volver a cargarla; cuando nadie la usa, desaparece de la caché sola. Registrar con `weakref.finalize` un aviso cuando se libere cada imagen. Comprobar con `gc.collect()` que la caché se vacía.

In [18]:
import gc
import weakref


class Imagen:
    def __init__(self, nombre):
        self.nombre = nombre


cache = weakref.WeakValueDictionary()
cargas, liberadas = [], []


def obtener(nombre):
    imagen = cache.get(nombre)
    if imagen is None:
        imagen = Imagen(nombre)
        cargas.append(nombre)
        weakref.finalize(imagen, liberadas.append, nombre)
        cache[nombre] = imagen
    return imagen


a = obtener("playa.png")
b = obtener("playa.png")
c = obtener("volcan.png")
print("Misma imagen:", a is b, "| cargas:", cargas, "| en caché:", sorted(cache))
assert a is b and cargas == ["playa.png", "volcan.png"]

del a, b
gc.collect()
print("Tras soltar playa.png -> en caché:", sorted(cache), "| liberadas:", liberadas)
assert sorted(cache) == ["volcan.png"] and liberadas == ["playa.png"]

del c
gc.collect()
print("Al final -> en caché:", sorted(cache), "| liberadas:", liberadas)
assert len(cache) == 0 and liberadas == ["playa.png", "volcan.png"]

Misma imagen: True | cargas: ['playa.png', 'volcan.png'] | en caché: ['playa.png', 'volcan.png']
Tras soltar playa.png -> en caché: ['volcan.png'] | liberadas: ['playa.png']
Al final -> en caché: [] | liberadas: ['playa.png', 'volcan.png']
